# STAT 301 Project — Online Shoppers Purchasing Intention

**Name:** Samika, Shrey, Ivy, Wilson 
**Group:**  Group-25   
---

In [1]:
# loading the appropriate libraries and packages
library(tidyverse) 
library(broom)
require(gridExtra)
library(leaps)
library(car) 
library(forcats)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: gridExtra


Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine


Loading required package: carData


Attaching package: ‘car’


The following object is masked from ‘package:dplyr’:

    recode


The following object is masked from ‘package:purrr’:

    some




## Introduction

Online shopping platforms routinely track users’ browsing behaviour to better understand what drives purchase decisions. In this project, we investigate which aspects of a visitor’s on-site behaviour are most strongly associated with completing a purchase. Using the Online Shoppers Purchasing Intention dataset, our goal is to determine how different behavioural indicators — such as the number of pages viewed, the value of pages visited, exit behaviour, and features of the visit itself — relate to the probability that a user ultimately buys a product.

Our response variable is Revenue, a binary indicator of whether the session resulted in a purchase (“Purchase”) or not (“No Purchase”). To explore behavioural factors that may influence purchasing likelihood, we focus on predictors that capture three major dimensions of online activity:

1. **Browsing depth** (*Administrative*, *Informational*, *ProductRelated*)
2. **Engagement quality** (*ExitRates*, *PageValues*)
3. **Session timing and visitor profile** (*SpecialDay*, *Weekend*, *VisitorType_simple*)

The aim of this project is inferential — rather than predicting purchases, we seek to understand how and to what extent these behavioural factors are associated with purchasing decisions. To address this research question, we use logistic regression, an appropriate method for modelling binary outcomes while enabling clear interpretation of how each explanatory variable affects the odds of purchasing.

## Data Description

### 1. Descriptive Summary

This data contains information about user behavior when online shopping. Each row in the dataset represents one online shopping session, and the variables describe various aspects of user behavior, technical attributes, and timing information. The dataset includes both quantitative and categorical variables, such as the number and duration of pages visited, bounce and exit rates, month of visit, type of operating system, and whether the session occurred on a weekend.  

- **Number of observations:** 12330  
- **Number of variables:** 18
#### Variable Description
  | Variable Name | Type | Description |
|-----------------------------|------------------|-------------------------------------------------------------|
| Administrative |Numeric(Integer) | Number of administrative pages visited by the user. |
| Informational | Numeric(Integer) | Number of informational pages visited by the user. |
| ProductRelated | Numeric(Integer) | Number of product-related pages visited by the user. |
| BounceRates | Numeric(Continuous) | Percentage of visitors who enter the site from that page and then leave ("bounce") without triggering any other requests to the analytics server during that session. |
| ExitRates | Numeric(Continuous) | Percentage of this page being the last session. |
| PageValues | Numeric(Integer) | Average value for a web page that a user visited before completing an e-commerce transaction.  |
| SpecialDay | Numeric(Integer) | Closeness of the site visit date to a special day (e.g., Mother’s Day, Christmas). Ranges from 0 to 1. |
| VisitorType | Categorical | Indicates if the visitor is a Returning Visitor or New Visitor. |
| Weekend | Binary (True/False) | Indicates whether the visit occurred on a weekend. |
| Revenue | Binary (0/1) | Indicates whether the session ended with a purchase (1 = purchase, 0 = no purchase). |

---


### 2. Source and Information 
This dataset is the **Online Shoppers Purchasing Intention** dataset published by UCI Machine Learning Repository. 
It contains session-level features for e-commerce website visits and a binary outcome `Revenue` indicating whether a purchase occurred. 

### 3. Variable Transformations
To prepare the dataset for modelling, several preprocessing steps were completed in Stage 2. Rows with missing or invalid `Region` values were removed to ensure the data represented valid sessions. The response variable `Revenue` was converted into a clearly labelled factor to support logistic regression. The `Weekend` variable was recoded into “Weekday” and “Weekend” to simplify interpretation. The `VisitorType` variable originally contained many small categories so it was simplified into “Returning Visitor,” “New Visitor,” and “Other” using factor lumping. This improves the model's stability and avoids sparse categorical levels. These transformations ensure variables are in the appropriate format for modelling and help improve interpretability and reliability of the final regression model.

In [2]:
data <- read_csv("") |>
        filter(!is.na(Region), Region != 1) |>
        mutate(Revenue = factor(Revenue,
                                levels = c(FALSE, TRUE),
                                labels = c("No Purchase", "Purchase")),
               Weekend = factor(Weekend,
                                levels = c(FALSE, TRUE),
                                labels = c("Weekday", "Weekend")),
               VisitorType_simple = fct_lump_n(VisitorType, n = 2, other_level = "Other"),
               VisitorType_simple = fct_relevel(VisitorType_simple, "Returning_Visitor")

ERROR: Error in parse(text = x, srcfile = src): <text>:11:0: unexpected end of input
9:                VisitorType_simple = fct_lump_n(VisitorType, n = 2, other_level = "Other"),
10:                VisitorType_simple = fct_relevel(VisitorType_simple, "Returning_Visitor")
   ^


## Scientific Question

Final Question: **How are key aspects of a user’s browsing depth, exit behaviour, timing and profile are associated with the probability that an online shopping session results in a purchase?**

### 1. Response Variable

The response variable is **Revenue**, which indicates whether the online shopping session ended with a purchase:

- `Revenue = "Purchase"`: the session resulted in a purchase.  
- `Revenue = "No Purchase"`: the session did not result in a purchase.

This binary outcome makes logistic regression an appropriate modelling choice.

### 2. Aim: Prediction, Inference, or Both?

The primary aim of this project is **inference**. Rather than building a model to predict purchases for new sessions, we are interested in understanding **how** the selected behavioural and timing variables are associated with the odds of making a purchase. Our goal is to quantify and interpret these associations (for example, how changes in `PageValues` or `ExitRates` relate to the odds of a purchase), while recognizing that the observational nature of the data limits causal conclusions.

### 3. Explanatory Variable Selection
To decide which explanatory variables to include in the final model, in stage 2, we first considered a set of numeric predictors that capture browsing depth, exit behaviour, and timing: `Administrative`, `Administrative_Duration`, `Informational`, `Informational_Duration`, `ProductRelated`, `ProductRelated_Duration`, `BounceRates`, `ExitRates`, `PageValues`, and `SpecialDay`. We fitted an initial logistic regression using all of these numeric variables and examined the coefficient patterns, p-values, a correlation matrix, and variance inflation factors (VIFs).

The correlation matrix showed that each page-count variable was strongly correlated with its corresponding duration variable (for example, `ProductRelated` vs. `ProductRelated_Duration`). Because these pairs describe the same underlying behaviour, we kept the page-count variables (`Administrative`, `Informational`, `ProductRelated`) and removed their duration counterparts to avoid redundancy and multicollinearity. A moderate correlation was also observed between `BounceRates` and `ExitRates`; we retained `ExitRates` as a broader measure of page exits and dropped `BounceRates`. In contrast, `PageValues` and `SpecialDay` had low correlations with other predictors and VIFs close to 1, indicating that they provide unique explanatory information.

After these checks, the final numeric set used in our model is:
`Administrative`, `Informational`, `ProductRelated`, `ExitRates`, `PageValues`, and `SpecialDay`. We then added the categorical predictors `Weekend` and `VisitorType_simple` to account for timing (weekday vs. weekend sessions) and visitor profile (returning vs. new or other visitors). All variables in the final model have VIF values below 5, suggesting that multicollinearity is not a major concern.



In [3]:
# Candidate numeric predictors considered in Stage 2
cand_num_vars <- c(
  "Administrative", "Administrative_Duration",
  "Informational", "Informational_Duration",
  "ProductRelated", "ProductRelated_Duration",
  "BounceRates", "ExitRates",
  "PageValues", "SpecialDay"
)

# Fit the initial numeric-only logistic regression model
full_formula_num <- as.formula(
  paste("Revenue ~", paste(cand_num_vars, collapse = " + "))
)

full_glm_num <- glm(full_formula_num, data = data, family = "binomial")

# ----- Table 1: Coefficient estimates -----
summary(full_glm_num)

# ----- Table 2: Correlation matrix for numeric predictors -----
corr_matrix <- data %>%
  select(all_of(cand_num_vars)) %>%
  cor()

corr_matrix

# ----- Table 3: VIFs for the numeric-only model -----
full_vif_num <- vif(full_glm_num)
full_vif_num

ERROR: Error in model.frame.default(formula = full_formula_num, data = data, : 'data' must be a data.frame, environment, or list


## EDA Visualisations 

In [9]:
## Samika plot

x_limit <- quantile(data$ProductRelated, 0.99, na.rm = TRUE)
y_limit <- quantile(data$ExitRates, 0.99, na.rm = TRUE)

eda_plot <- ggplot(
  data,
  aes(x = ProductRelated, y = ExitRates, colour = Revenue)
) +
  geom_point(alpha = 0.25, size = 1) +
  geom_smooth(
    method = "loess",
    se = FALSE,
    linewidth = 1
  ) +
  facet_grid(
    rows = vars(NearSpecialDay),
    cols = vars(VisitorType_simple)
  ) +
  coord_cartesian(
    xlim = c(0, x_limit),
    ylim = c(0, y_limit)
  ) +
  labs(
    title = "Product Exploration vs Exit Rates by Purchase Outcome",
    subtitle = "Faceted by special shopping days and visitor type",
    x = "Number of product-related pages viewed (ProductRelated)",
    y = "Average exit rate from visited pages (ExitRates)",
    colour = "Purchase outcome"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    plot.title = element_text(face = "bold", size = 14),
    plot.subtitle = element_text(size = 11),
    strip.text = element_text(face = "bold"),
    legend.position = "bottom",
    panel.grid.minor = element_blank()
  ) +
  guides(
    colour = guide_legend(
      override.aes = list(alpha = 1, size = 3)
    )
  )

eda_plot

ERROR: Error in data$ProductRelated: object of type 'closure' is not subsettable


In [ ]:
## Wilson plot 

## Methodology

### 1. Step-by-step procedure

1. **Load and preprocess the data.**  
   We started from the provided `data` object (Online Shoppers Purchasing Intention dataset). We removed sessions with missing or invalid `Region` values, converted `Revenue` into a labelled factor (“No Purchase”, “Purchase”), recoded `Weekend` into “Weekday” vs “Weekend”, and simplified `VisitorType` into `VisitorType_simple` with three levels (“Returning_Visitor”, “New_Visitor”, “Other”) using factor lumping. These transformations ensured that the response and categorical predictors were in the correct format for logistic regression and that sparse categories were combined.

2. **Exploratory data analysis.**  
   We created a main visualization of `ProductRelated` (number of product-related pages) versus `ExitRates`, coloured by `Revenue` and faceted by special-day indicator and visitor type. This plot allowed us to visually assess how product exploration and exit behaviour differ between purchase and non-purchase sessions across different types of visitors and timing.

3. **Initial numeric-only logistic regression and diagnostics (Stage 2).**  
   Focusing on numeric predictors only, we fitted a full logistic regression model with `Revenue` as the response and all candidate numeric variables (page counts, durations, exit rates, page values, and special-day proximity) as predictors. From this model, we examined coefficient estimates and p-values, computed a correlation matrix for the numeric predictors, and calculated variance inflation factors (VIFs) to diagnose multicollinearity.

4. **Selection of numeric explanatory variables (Stage 2).**  
   Using the coefficient patterns, correlations, and VIFs, we removed redundant predictors. Duration variables were dropped because they were highly correlated with their corresponding page-count variables (e.g., `ProductRelated` vs `ProductRelated_Duration`) and did not add distinct explanatory power. Between `BounceRates` and `ExitRates`, we retained `ExitRates` because it provides a broader measure of page exits and had acceptable VIFs. The final numeric set consisted of `Administrative`, `Informational`, `ProductRelated`, `ExitRates`, `PageValues`, and `SpecialDay`.

5. **Build the final logistic regression model.**  
   We then constructed a combined logistic regression model including both the selected numeric predictors and the categorical predictors `Weekend` and `VisitorType_simple`. The response variable remained `Revenue` (Purchase vs No Purchase). This final model relates the log-odds of a purchase to browsing depth (page counts), exit behaviour, the value of visited pages, timing relative to special days, and visitor profile, using the `glm()` function with a binomial family.

6. **Model diagnostics for the final model.**  
   For the final model, we re-computed VIFs to confirm that multicollinearity is not severe (all VIFs < 5). We compared the null deviance to the residual deviance to check that the full model provides a substantial improvement over an intercept-only model. Optionally, we also inspected a classification summary (predicted vs actual `Revenue`) to ensure that the model produces reasonable fitted probabilities.

7. **Results summarisation and interpretation.**  
   Finally, we extracted the estimated coefficients and p-values, and also exponentiated the coefficients to obtain odds ratios. We focused our interpretation on statistically significant predictors (e.g., `PageValues`, `ExitRates`, `SpecialDay`, and visitor type), explaining in plain language how changes in each variable are associated with the odds of a purchase while holding other variables fixed. These results were then linked back to the original scientific question about how browsing behaviour, exit behaviour, and visit timing relate to purchasing.

### 2. Justification

This methodology follows the plan from Stage 2. Because `Revenue` is a binary outcome, logistic regression is an appropriate choice for modelling how explanatory variables affect purchase probability. Starting with exploratory visualisation helps us understand key relationships and potential non-linearities. The staged variable-selection process—fitting a full numeric model, examining correlations and VIFs, and then adding categorical predictors—ensures that the final model includes non-redundant, interpretable predictors while controlling for multicollinearity. Checking VIFs and comparing null vs residual deviance provides basic model diagnostics consistent with the course. The final step of interpreting coefficients and odds ratios directly addresses our inferential goal: understanding which aspects of user behaviour and timing are associated with online purchases.

### 3. Assumptions

The main assumptions underlying this analysis are:

- **Independent observations:** Each session is treated as an independent visit; one user’s behaviour does not directly affect another’s within the dataset.
- **Logistic regression form:** The log-odds of a purchase are assumed to be a linear function of the predictors included in the model.
- **No severe multicollinearity:** Predictors are not highly collinear, which we checked using correlation matrices and VIFs.
- **Correct model specification:** The chosen predictors capture the main behavioural and timing effects relevant to purchase outcome, and the logit link is appropriate for a binary response.

### 4. Limitations

The dataset is observational, so our model can identify associations but cannot establish causation. Important factors such as marketing campaigns, user demographics, or device type may be missing, leading to potential omitted-variable bias. Although we reduced multicollinearity, some predictors may still be correlated, which can affect the stability of individual coefficient estimates. Finally, the data come from a specific e-commerce context, so the results may not generalise to all online retailers or time periods.


## Final Model

### 2. Model equation

Let \(Y\) be the indicator that the session results in a purchase,  
so \(Y = 1\) if `Revenue = "Purchase"` and \(Y = 0\) if `Revenue = "No Purchase"`.

The logistic regression model is

log-odds(Purchase) =
β₀
+ β₁·Administrative
+ β₂·Informational
+ β₃·ProductRelated
+ β₄·ExitRates
+ β₅·PageValues
+ β₆·SpecialDay
+ β₇·I(Weekend = "Weekend")
+ β₈·I(VisitorType_simple = "New_Visitor")
+ β₉·I(VisitorType_simple = "Other").

Here:

- `Weekend = "Weekday"` is the baseline level for **Weekend**.
- `VisitorType_simple = "Returning_Visitor"` is the baseline level for **VisitorType_simple**.

Each coefficient \(\beta_j\) describes the change in the log-odds of a purchase associated with a one-unit increase in the corresponding numeric predictor, or with switching from the baseline category to the indicated level for categorical predictors, holding all other variables fixed.

### 3. Implementation of the final model


In [10]:
final_glm <- glm(
  Revenue ~ Administrative + Informational + ProductRelated +
    ExitRates + PageValues + SpecialDay +
    Weekend + VisitorType_simple,
  data = data,
  family = "binomial"
)

ERROR: Error in model.frame.default(formula = Revenue ~ Administrative + Informational + : 'data' must be a data.frame, environment, or list


## Model Diagnostics

In [6]:
edf

ERROR: Error in eval(expr, envir, enclos): object 'edf' not found


## Results & Interpretation

In [7]:
fr

ERROR: Error in eval(expr, envir, enclos): object 'fr' not found
